# YOLOv8 Training, Export to TFLite, and Depth-Proxy Evaluation

The purpose of this notebook is to:
1. Validate the existing model
2. Merge new Roboflow data (with class-ID remapping)
3. Train and validate YOLOv8
4. Export the best model to **TFLite** for Android/React Native
5. Run a **depth proxy** experiment (work-in-progress)


## 1) Environment setup
**Steps**
1. Clone the repo
2. Enter the repo directory
3. Install Ultralytics


In [ ]:
# Clone your forked repo into the colab notebook (change the link accordingly)
!git clone https://github.com/rkorlahalli/AI-Assisted-Navigation-Device_Test

%cd AI-Assisted-Navigation-Device_Test
!ls


Cloning into 'AI-Assisted-Navigation-Device_Test'...
remote: Enumerating objects: 6240, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 6240 (delta 65), reused 59 (delta 59), pack-reused 6144 (from 1)
Receiving objects: 100% (6240/6240), 958.39 MiB | 35.13 MiB/s, done.
Resolving deltas: 100% (262/262), done.
Updating files: 100% (1964/1964), done.
/content/AI-Assisted-Navigation-Device_Test
Documentation_side  Hardware_side  image.png  ML_side  README.md  software_side


In [ ]:
cd /content/AI-Assisted-Navigation-Device_Test

/content/AI-Assisted-Navigation-Device_Test


In [ ]:
%pip install ultralytics


## 2) Baseline validation (optional)
Use this section if you want to re-evaluate the *current* model before merging new data.


In [ ]:
# Use this to validate model beforing merging new data

from ultralytics import YOLO

model = YOLO("ML_side/models/best.pt")

results = model.val(
    data="ML_side/config/newdata.yaml",
    imgsz=640,
)

metrics = results.results_dict
print("Precision:    ", metrics["metrics/precision(B)"])
print("Recall:       ", metrics["metrics/recall(B)"])
print("mAP@0.5:      ", metrics["metrics/mAP50(B)"])
print("mAP@0.5:0.95: ", metrics["metrics/mAP50-95(B)"])


## 3) Import Roboflow export
**Steps**
1. Upload the downloaded Roboflow dataset zip to Colab
2. Unzip to `/content/roboflow_data`
3. Verify the folder structure


In [ ]:
# Upload downloaded dataset first then unzip

!unzip -q "/content/AIAND_Main.v1i.yolov8.zip" -d "/content/roboflow_data"
!ls /content/roboflow_data


In [ ]:
# Check Roboflow export
!ls /content/roboflow_data

# Check repo
!ls /content
!ls /content/AI-Assisted-Navigation-Device_Test


## 4) Remap Roboflow class IDs to match the existing dataset
**Steps**
1. Configure `id_map`
2. Remap label files in `train` and `valid`
3. Spot-check a few label files


In [ ]:
# Use to remap id's from Roboflow export to align with existing dataset IDs

import pathlib

# Root of Roboflow export in Colab (contains train/valid folders)
root = pathlib.Path("/content/roboflow_data")

# Roboflow 4-class export order:
# 0 Couch, 1 Table, 2 Television, 3 Whiteboard
# Desired 8-class IDs:
# whiteboard=4, table=5, tv=6, couch=7
id_map = {
    0: 7,  # Couch -> couch
    1: 5,  # Table -> table
    2: 6,  # Television -> tv
    3: 4,  # Whiteboard -> whiteboard
}

def remap_split(split_name: str):
    labels_dir = root / split_name / "labels"
    if not labels_dir.exists():
        print(f"Skipping {split_name}, no labels dir at {labels_dir}")
        return

    changed = 0
    unknown = 0

    for txt_path in labels_dir.glob("*.txt"):
        new_lines = []
        with txt_path.open("r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                old_id = int(float(parts[0]))

                if old_id not in id_map:
                    unknown += 1
                    # keep the line unchanged (or raise error if you prefer)
                    new_lines.append(" ".join(parts))
                    continue

                parts[0] = str(id_map[old_id])
                new_lines.append(" ".join(parts))
                changed += 1

        txt_path.write_text("\n".join(new_lines) + "\n")

    print(f"{split_name}: remapped {changed} objects. Unknown IDs left unchanged: {unknown}")

for split in ["train", "valid", "test"]:
    remap_split(split)

print("Done remapping IDs.")


In [ ]:
!head /content/roboflow_data/train/labels/*.txt | head


## 5) Merge Roboflow data into the repo dataset
**Steps**
1. Copy Roboflow `train` images/labels into the repo train folders
2. Copy Roboflow `valid` images/labels into the repo validation folders
3. Verify files exist after copy
4. Check class instance counts


In [ ]:
ROBO = "/content/roboflow_data"

# Append Roboflow training images + labels
!cp -r "$ROBO/train/images/." ML_side/data/processed/train_dataset/train/images/
!cp -r "$ROBO/train/labels/." ML_side/data/processed/train_dataset/train/labels/


In [ ]:
# Append Roboflow validation images + labels
!cp -r "$ROBO/valid/images/." ML_side/data/processed/val_dataset/val/images/
!cp -r "$ROBO/valid/labels/." ML_side/data/processed/val_dataset/val/labels/


In [ ]:
!ls ML_side/data/processed/train_dataset/train/images | head
!ls ML_side/data/processed/val_dataset/val/images   | head

!ls ML_side/data/processed/train_dataset/train/images | wc -l
!ls ML_side/data/processed/val_dataset/val/images   | wc -l


In [ ]:
# Use this to check the number of instances in each class

import os, glob
from collections import Counter, defaultdict

REPO_ROOT = "/content/AI-Assisted-Navigation-Device_Test"
TRAIN_LBL = os.path.join(REPO_ROOT, "ML_side/data/processed/train_dataset/train/labels")
VAL_LBL   = os.path.join(REPO_ROOT, "ML_side/data/processed/val_dataset/val/labels")

CLASS_NAMES = ["book","books","monitor","office-chair","whiteboard","table","tv","couch"]

def scan_labels(label_dir):
    inst = Counter()
    imgs = defaultdict(set)
    bad_ids = Counter()

    for p in glob.glob(os.path.join(label_dir, "*.txt")):
        fname = os.path.basename(p)
        with open(p, "r") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        for ln in lines:
            parts = ln.split()
            if len(parts) < 1:
                continue
            cid = int(float(parts[0]))
            if cid < 0 or cid >= len(CLASS_NAMES):
                bad_ids[cid] += 1
                continue
            inst[cid] += 1
            imgs[cid].add(fname)

    img_counts = {cid: len(s) for cid, s in imgs.items()}
    return inst, img_counts, bad_ids

train_inst, train_imgs, train_bad = scan_labels(TRAIN_LBL)
val_inst, val_imgs, val_bad = scan_labels(VAL_LBL)

print("TRAIN instances:", dict(train_inst))
print("TRAIN images   :", {CLASS_NAMES[k]: v for k,v in sorted(train_imgs.items())})
print("TRAIN bad IDs  :", dict(train_bad))

print("\nVAL instances  :", dict(val_inst))
print("VAL images     :", {CLASS_NAMES[k]: v for k,v in sorted(val_imgs.items())})
print("VAL bad IDs    :", dict(val_bad))


## 6) Dataset configuration (YAML)
Write/update the dataset YAML used for training and validation.


In [ ]:
# Write yaml file with new class

%%writefile ML_side/config/newdata.yaml
path: ML_side/data/processed

train: train_dataset/train/images
val:   val_dataset/val/images

nc: 8
names:
  - book
  - books
  - monitor
  - office-chair
  - whiteboard
  - table
  - tv
  - couch


## 7) Train YOLOv8
**Steps**
1. Confirm paths exist (repo/model/yaml)
2. Train the model using `model.train(...)`


In [ ]:
# Set variables and path for repo, model, and yaml and check if they exist

from pathlib import Path

REPO = Path("/content/AI-Assisted-Navigation-Device_Test")

MODEL_PT  = REPO / "ML_side/models/best.pt"
DATA_YAML = REPO / "ML_side/config/newdata.yaml"

print("MODEL:", MODEL_PT.exists(), MODEL_PT)
print("YAML :", DATA_YAML.exists(), DATA_YAML)


In [ ]:
# Train model as per existing data config

from ultralytics import YOLO

model = YOLO("ML_side/models/best.pt")

results = model.train(
    data="ML_side/config/newdata.yaml",
    imgsz=640,
    epochs=100,
    batch=16,
    optimizer="Adam",
    lr0=0.003,
    workers=2,
    device=0,
    cache=False,
    amp=True,
    patience=20,
)


## 8) Validate the best checkpoint
This section finds the latest `best.pt` from the training run and evaluates it.


In [ ]:
# Set variable path to best saved model and validate results

import glob
from ultralytics import YOLO

best_path = sorted(glob.glob("runs/detect/train*/weights/best.pt"))[-1]
print("Using:", best_path)

best_model = YOLO(best_path)
val_results = best_model.val(data=str(DATA_YAML), imgsz=640, device=0)

print(val_results.results_dict)

## 9) Export to TFLite and download
**Steps**
1. Export the best model to TFLite
2. Download the exported `.tflite` file


In [ ]:
# Export best saved model to tflite format so it can deploy with react native on android

best_model.export(format="tflite", imgsz=640, nms=True)
best_model.export(format="tflite", imgsz=640, nms=True, half=True)


In [ ]:
from google.colab import files
files.download("/content/AI-Assisted-Navigation-Device_Test/runs/detect/train/weights/best_saved_model/best_float16.tflite")


## 10) TFLite validation (inference + visualisation)
Runs inference using the TFLite interpreter and visualises predictions.
> Note: Full mAP computation from TFLite outputs requires consistent parsing of the output tensor format.


In [ ]:
!pip -q install pycocotools opencv-python
!pip -q install pycocotools opencv-python
!pip -q install tensorflow


In [ ]:
TFLITE_PATH = "/content/AI-Assisted-Navigation-Device_Test/runs/detect/train/weights/best_saved_model/best_float16.tflite"

VAL_IMG_DIR = "/content/AI-Assisted-Navigation-Device_Test/ML_side/data/processed/val_dataset/val/images"
VAL_LBL_DIR = "/content/AI-Assisted-Navigation-Device_Test/ML_side/data/processed/val_dataset/val/labels"

CLASS_NAMES = ["book","books","monitor","office-chair","whiteboard","table","tv","couch"]


In [ ]:
import os, glob
import numpy as np
import cv2

# Try tflite-runtime first, fallback to TF
try:
    from tflite_runtime.interpreter import Interpreter
except ImportError:
    from tensorflow.lite.python.interpreter import Interpreter

OUT_PRED_DIR = "/content/tflite_preds"
os.makedirs(OUT_PRED_DIR, exist_ok=True)

interpreter = Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

in_details = interpreter.get_input_details()
out_details = interpreter.get_output_details()

inp_h, inp_w = in_details[0]["shape"][1], in_details[0]["shape"][2]

def preprocess(img_bgr):
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (inp_w, inp_h))
    x = img.astype(np.float32) / 255.0
    x = np.expand_dims(x, axis=0)
    return x

def run_tflite(img_bgr):
    x = preprocess(img_bgr)
    interpreter.set_tensor(in_details[0]["index"], x)
    interpreter.invoke()
    outputs = [interpreter.get_tensor(o["index"]) for o in out_details]
    return outputs

def save_preds_yolo(img_path, outputs, conf_thres=0.25):
    img0 = cv2.imread(img_path)
    h0, w0 = img0.shape[:2]

    det = outputs[0]
    det = np.squeeze(det)  # [N,6] or similar

    lines = []
    if det.ndim == 1:
        det = det.reshape(1, -1)

    for row in det:
        if row.shape[0] < 6:
            continue
        x1, y1, x2, y2, score, cls = row[:6]
        score = float(score)
        if score < conf_thres:
            continue
        cls = int(cls)

        # convert to YOLO normalized xywh
        bw = max(0.0, x2 - x1)
        bh = max(0.0, y2 - y1)
        cx = x1 + bw / 2.0
        cy = y1 + bh / 2.0

        # normalize by original image size
        cx /= w0; cy /= h0; bw /= w0; bh /= h0

        lines.append(f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f} {score:.6f}")

    out_txt = os.path.join(OUT_PRED_DIR, os.path.splitext(os.path.basename(img_path))[0] + ".txt")
    with open(out_txt, "w") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))

# Run on all val images
val_imgs = sorted(glob.glob(os.path.join(VAL_IMG_DIR, "*.*")))
print("Val images:", len(val_imgs))

for i, p in enumerate(val_imgs):
    img = cv2.imread(p)
    outs = run_tflite(img)
    save_preds_yolo(p, outs, conf_thres=0.25)

print("Saved predictions to:", OUT_PRED_DIR)


In [ ]:
# Visualise predictions

import random, matplotlib.pyplot as plt

def draw_yolo_preds(img_path, pred_txt):
    img = cv2.imread(img_path)
    h,w = img.shape[:2]
    if os.path.exists(pred_txt):
        for ln in open(pred_txt):
            parts = ln.split()
            if len(parts) < 6:
                continue
            cls, cx, cy, bw, bh, conf = parts
            cx=float(cx)*w; cy=float(cy)*h; bw=float(bw)*w; bh=float(bh)*h
            x1=int(cx-bw/2); y1=int(cy-bh/2); x2=int(cx+bw/2); y2=int(cy+bh/2)
            cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(img, f"{CLASS_NAMES[int(cls)]}:{float(conf):.2f}", (x1, max(15,y1-5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

samples = random.sample(val_imgs, min(6, len(val_imgs)))
plt.figure(figsize=(12,8))
for i,p in enumerate(samples,1):
    pred = os.path.join(OUT_PRED_DIR, os.path.splitext(os.path.basename(p))[0]+".txt")
    plt.subplot(2,3,i)
    plt.imshow(draw_yolo_preds(p, pred))
    plt.axis("off")
plt.show()


## 11) Depth estimation (work in progress)
This section implements a **depth proxy** experiment to explore how depth cues might be integrated.
It does **not** implement real metric depth estimation for mobile yet.


### 11.1 Install dependencies


In [ ]:

!pip -q install opencv-python-headless pandas

### 11.2 Load paths and prepare image list


In [ ]:
import os, glob
import pandas as pd
import numpy as np
import cv2

# Set repo
REPO_ROOT = "/content/AI-Assisted-Navigation-Device_Test"

# Validation dataset paths
VAL_IMG_DIR = os.path.join(REPO_ROOT, "ML_side/data/processed/val_dataset/val/images")
VAL_LBL_DIR = os.path.join(REPO_ROOT, "ML_side/data/processed/val_dataset/val/labels")

print("VAL_IMG_DIR:", VAL_IMG_DIR)
print("VAL_LBL_DIR:", VAL_LBL_DIR)

img_paths = sorted(glob.glob(os.path.join(VAL_IMG_DIR, "*.*")))
print("Validation images found:", len(img_paths))

CLASS_NAMES = ["book","books","monitor","office-chair","whiteboard","table","tv","couch"]
print("Class order:", CLASS_NAMES)

### 11.3 Baseline depth proxy function


In [ ]:
def baseline_depth_score(box_height_px: float, img_height_px: int) -> float:
    """Baseline depth proxy: larger boxes are assumed closer."""
    return float(box_height_px) / float(img_height_px)

def improved_depth_score(box_height_px: float, box_center_y_px: float, img_height_px: int,
                         alpha: float = 0.7, beta: float = 0.3) -> float:
    """Improved proxy: combine size and vertical position cues."""
    size_score = float(box_height_px) / float(img_height_px)
    position_score = float(box_center_y_px) / float(img_height_px)
    return alpha * size_score + beta * position_score

def read_yolo_labels(label_path: str):
    """Read YOLO txt labels: class x_center y_center width height (all normalized)."""
    rows = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls = int(float(parts[0]))
            xc, yc, w, h = map(float, parts[1:])
            rows.append((cls, xc, yc, w, h))
    return rows

### 11.4 Compute depth proxy scores from labels


In [ ]:
# Computes depth scores from label files, this is used only as a proxy
records = []

for img_path in img_paths:
    base = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(VAL_LBL_DIR, base + ".txt")
    if not os.path.exists(lbl_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue
    H, W = img.shape[:2]

    labels = read_yolo_labels(lbl_path)
    for cls, xc, yc, w, h in labels:
        # Convert normalized YOLO dims to pixels
        box_h_px = h * H
        box_center_y_px = yc * H

        records.append({
            "image": os.path.basename(img_path),
            "class_id": cls,
            "class_name": CLASS_NAMES[cls] if 0 <= cls < len(CLASS_NAMES) else "unknown",
            "box_h_px": box_h_px,
            "baseline_depth_score": baseline_depth_score(box_h_px, H),
            "improved_depth_score": improved_depth_score(box_h_px, box_center_y_px, H),
        })

df = pd.DataFrame(records)
print("Label-based depth records:", len(df))
df.head()

### 11.5 Save proxy results and summarise by class


In [ ]:
# Save CSV
out_csv = os.path.join(REPO_ROOT, "ML_side/docs/depth_proxy_validation.csv")
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [ ]:
# Summary by class (counts + descriptive stats)
summary = (
    df.groupby("class_name")[["baseline_depth_score","improved_depth_score"]]
      .agg(["count","mean","std","min","max"])
      .sort_values(("baseline_depth_score","count"), ascending=False)
)
summary

### 11.6 Generate predictions and save depth proxy outputs


In [ ]:
from ultralytics import YOLO

# Load trained model
MODEL_PATH = os.path.join(REPO_ROOT, "ML_side/models/best.pt")
model = YOLO(MODEL_PATH)

sample_imgs = img_paths[:50]  # adjust to trade speed vs coverage
pred_records = []

for img_path in sample_imgs:
    img = cv2.imread(img_path)
    if img is None:
        continue
    H, W = img.shape[:2]

    # predict
    r = model.predict(img_path, conf=0.25, iou=0.7, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        continue

    for b in r.boxes:
        cls = int(b.cls.item())
        conf = float(b.conf.item())
        x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())

        box_h_px = (y2 - y1)
        box_center_y_px = (y1 + y2) / 2.0

        pred_records.append({
            "image": os.path.basename(img_path),
            "class_id": cls,
            "class_name": CLASS_NAMES[cls] if 0 <= cls < len(CLASS_NAMES) else "unknown",
            "yolo_conf": conf,
            "box_h_px": box_h_px,
            "baseline_depth_score": baseline_depth_score(box_h_px, H),
            "improved_depth_score": improved_depth_score(box_h_px, box_center_y_px, H),
        })

pred_df = pd.DataFrame(pred_records)
print("Prediction-based depth records:", len(pred_df))
pred_df.head()

In [ ]:
# Save prediction CSV
pred_csv = os.path.join(REPO_ROOT, "ML_side/docs/depth_proxy_predictions.csv")
os.makedirs(os.path.dirname(pred_csv), exist_ok=True)
pred_df.to_csv(pred_csv, index=False)
print("Saved:", pred_csv)